In [13]:
import sqlite3
import pandas as pd
import numpy as np


conn=sqlite3.connect(r"C:\Users\KRISH JHA\OneDrive\Desktop\Sepsis\model\data_handling\sepsis.db")

cursor=conn.cursor()

print("Database is connnected")


Database is connnected


In [3]:
df=pd.read_sql_query("""
Select * from clean_patient_data;
""",conn)

df.columns()

Index(['Patient_id', 'HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp',
       'BaseExcess', 'FiO2', 'pH', 'PaCO2', 'BUN', 'Calcium', 'Creatinine',
       'Glucose', 'Magnesium', 'Potassium', 'Hct', 'Hgb', 'WBC', 'Platelets',
       'Age', 'Gender', 'Unit1', 'Unit2', 'HospAdmTime', 'ICULOS',
       'SepsisLabel', 'HR_missing', 'O2Sat_missing', 'Temp_missing',
       'SBP_missing', 'MAP_missing', 'DBP_missing', 'Resp_missing',
       'BaseExcess_missing', 'FiO2_missing', 'pH_missing', 'PaCO2_missing',
       'BUN_missing', 'Calcium_missing', 'Creatinine_missing',
       'Glucose_missing', 'Magnesium_missing', 'Potassium_missing',
       'Hct_missing', 'Hgb_missing', 'WBC_missing', 'Platelets_missing',
       'Unit1_missing', 'Unit2_missing', 'HospAdmTime_missing'],
      dtype='str')

In [15]:
vital_column={
     'HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp'
}



In [37]:
feature_table=df.copy()

feature_table=feature_table.sort_values(['Patient_id','ICULOS']).reset_index(drop=True)

windows=[3,6]
for col in vital_column:

    grouped = feature_table.groupby("Patient_id")[col]
    for wind in windows:

        feature_table[f'{col}_Rolling_Mean_{wind}h']=grouped.transform(lambda x:x.rolling(window=wind,min_periods=1).mean())
        feature_table[f'{col}_Rolling_Min_{wind}h']=grouped.transform(lambda x:x.rolling(window=wind,min_periods=1).min())
        feature_table[f'{col}_Rolling_Max_{wind}h']=grouped.transform(lambda x:x.rolling(window=wind,min_periods=1).max())
        feature_table[f'{col}_Rolling_Std_{wind}h']=grouped. transform(lambda x:x.rolling(window=wind,min_periods=1).std())
        


# feature_table['HR_Rolling_Mean']=df.groupby('Patient_id')['HR'].transform(lambda x:x.rolling(window=3,min_periods=1).mean())

feature_table.head(20)

,Patient_id,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,BaseExcess,FiO2,...,Temp_Rolling_Max_6h,Temp_Rolling_Std_6h,SBP_Rolling_Mean_3h,SBP_Rolling_Min_3h,SBP_Rolling_Max_3h,SBP_Rolling_Std_3h,SBP_Rolling_Mean_6h,SBP_Rolling_Min_6h,SBP_Rolling_Max_6h,SBP_Rolling_Std_6h
0,p000001,97.0,95.0,36.11,98.0,75.33,62.0,19.0,24.0,0.28,...,36.11,NaN,98.000000,98.0,98.0,NaN,98.000000,98.0,98.0,NaN
1,p000001,97.0,95.0,36.11,98.0,75.33,62.0,19.0,24.0,0.28,...,36.11,0.000000,98.000000,98.0,98.0,0.000000,98.000000,98.0,98.0,0.000000
2,p000001,89.0,99.0,36.11,122.0,86.00,62.0,22.0,24.0,0.28,...,36.11,0.000000,106.000000,98.0,122.0,13.856406,106.000000,98.0,122.0,13.856406
3,p000001,90.0,95.0,36.11,122.0,86.00,62.0,30.0,24.0,0.28,...,36.11,0.000000,114.000000,98.0,122.0,13.856406,110.000000,98.0,122.0,13.856406
4,p000001,103.0,88.5,36.11,122.0,91.33,62.0,24.5,24.0,0.28,...,36.11,0.000000,122.000000,122.0,122.0,0.000000,112.400000,98.0,122.0,13.145341
5,p000001,110.0,91.0,36.11,122.0,91.33,62.0,22.0,24.0,0.28,...,36.11,0.000000,122.000000,122.0,122.0,0.000000,114.000000,98.0,122.0,12.393547
6,p000001,108.0,92.0,36.11,123.0,77.00,62.0,29.0,24.0,0.28,...,36.11,0.000000,122.333333,122.0,123.0,0.577350,118.166667,98.0,123.0,9.887703
7,p000001,106.0,90.5,36.11,93.0,76.33,62.0,29.0,24.0,0.28,...,36.11,0.000000,112.666667,93.0,123.0,17.039171,117.333333,93.0,123.0,11.927559
8,p000001,104.0,95.0,36.11,133.0,88.33,62.0,26.0,24.0,0.28,...,36.11,0.000000,116.333333,93.0,133.0,20.816660,119.166667,93.0,133.0,13.526517
9,p000001,102.0,91.0,36.11,134.0,87.33,62.0,30.0,24.0,0.28,...,36.11,0.000000,120.000000,93.0,134.0,23.388031,121.166667,93.0,134.0,14.851487


In [31]:
feature_table=feature_table.drop(columns=['DBP_Rolling_Mean_3h', 'DBP_Rolling_Mean_6h', 'MAP_Rolling_Mean_3h',
       'MAP_Rolling_Mean_6h', 'HR_Rolling_Mean_3h', 'HR_Rolling_Mean_6h',
       'Resp_Rolling_Mean_3h', 'Resp_Rolling_Mean_6h', 'O2Sat_Rolling_Mean_3h',
       'O2Sat_Rolling_Mean_6h', 'Temp_Rolling_Mean_3h', 'Temp_Rolling_Mean_6h',
       'SBP_Rolling_Mean_3h', 'SBP_Rolling_Mean_6h'])

In [43]:
feature_table.shape
feature_table.isnull().sum()
feature_table.info()



<class 'pandas.DataFrame'>
RangeIndex: 1552210 entries, 0 to 1552209
Columns: 109 entries, Patient_id to SBP_Rolling_Std_6h
dtypes: float64(84), int64(24), str(1)
memory usage: 1.3 GB


In [45]:
feature_table.columns

Index(['Patient_id', 'HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp',
       'BaseExcess', 'FiO2',
       ...
       'Temp_Rolling_Max_6h', 'Temp_Rolling_Std_6h', 'SBP_Rolling_Mean_3h',
       'SBP_Rolling_Min_3h', 'SBP_Rolling_Max_3h', 'SBP_Rolling_Std_3h',
       'SBP_Rolling_Mean_6h', 'SBP_Rolling_Min_6h', 'SBP_Rolling_Max_6h',
       'SBP_Rolling_Std_6h'],
      dtype='str', length=109)

In [48]:
cols={'DBP_Rolling_Mean_3h','DBP_Rolling_Min_3h','DBP_Rolling_Max_3h','DBP_Rolling_Std_3h',
     'DBP_Rolling_Mean_6h','DBP_Rolling_Min_6h','DBP_Rolling_Max_6h','DBP_Rolling_Std_6h', 
     'MAP_Rolling_Mean_3h','MAP_Rolling_Min_3h','MAP_Rolling_Max_3h','MAP_Rolling_Std_3h', 
     'MAP_Rolling_Mean_6h','MAP_Rolling_Min_6h','MAP_Rolling_Max_6h','MAP_Rolling_Std_6h',
     'HR_Rolling_Mean_3h','HR_Rolling_Min_3h','HR_Rolling_Max_3h','HR_Rolling_Std_3h',
     'HR_Rolling_Mean_6h','HR_Rolling_Min_6h','HR_Rolling_Max_6h','HR_Rolling_Std_6h',
     'Resp_Rolling_Mean_3h','Resp_Rolling_Min_3h','Resp_Rolling_Max_3h','Resp_Rolling_Std_3h',
     'Resp_Rolling_Mean_6h','Resp_Rolling_Min_6h','Resp_Rolling_Max_6h','Resp_Rolling_Std_6h',
     'O2Sat_Rolling_Mean_3h', 'O2Sat_Rolling_Max_3h', 'O2Sat_Rolling_Min_3h', 'O2Sat_Rolling_Std_3h',
     'O2Sat_Rolling_Mean_6h', 'O2Sat_Rolling_Max_6h', 'O2Sat_Rolling_Min_6h', 'O2Sat_Rolling_Std_6h',
     'Temp_Rolling_Mean_3h','Temp_Rolling_Min_3h','Temp_Rolling_Max_3h', 'Temp_Rolling_Std_3h',
     'Temp_Rolling_Mean_6h','Temp_Rolling_Min_6h','Temp_Rolling_Max_6h', 'Temp_Rolling_Std_6h',
     'SBP_Rolling_Mean_3h','SBP_Rolling_Min_3h', 'SBP_Rolling_Max_3h', 'SBP_Rolling_Std_3h',
     'SBP_Rolling_Mean_6h', 'SBP_Rolling_Min_6h', 'SBP_Rolling_Max_6h','SBP_Rolling_Std_6h'}

In [52]:
engineered_columns = [
    col for col in feature_table.columns
    if "_Mean_" in col
    or "_Min_" in col
    or "_Max_" in col
    or "_Std_" in col
    or "_change_" in col
]

feature_table[engineered_columns] = (
    feature_table[engineered_columns]
    .fillna(0)
)

In [53]:
feature_table.isnull().sum()

Patient_id             0
HR                     0
O2Sat                  0
Temp                   0
SBP                    0
                      ..
SBP_Rolling_Std_3h     0
SBP_Rolling_Mean_6h    0
SBP_Rolling_Min_6h     0
SBP_Rolling_Max_6h     0
SBP_Rolling_Std_6h     0
Length: 109, dtype: int64

In [54]:
print(
    "Clean rows:",
    len(df)
)

print(
    "Feature rows:",
    len(feature_table)
)

Clean rows: 1552210
Feature rows: 1552210


In [55]:
missing = (
    feature_table
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print(missing[missing > 0])

Series([], dtype: int64)


In [56]:
feature_table.to_sql(
    "feature_table",
    conn,
    if_exists='replace',
    index_label=False
)

1552210

In [57]:
ft=pd.read_sql_query("""
 Select * from feature_table
""",conn)

ft.head()

,False,Patient_id,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,BaseExcess,...,Temp_Rolling_Max_6h,Temp_Rolling_Std_6h,SBP_Rolling_Mean_3h,SBP_Rolling_Min_3h,SBP_Rolling_Max_3h,SBP_Rolling_Std_3h,SBP_Rolling_Mean_6h,SBP_Rolling_Min_6h,SBP_Rolling_Max_6h,SBP_Rolling_Std_6h
0,0,p000001,97.0,95.0,36.11,98.0,75.33,62.0,19.0,24.0,...,36.11,0.0,98.0,98.0,98.0,0.000000,98.0,98.0,98.0,0.000000
1,1,p000001,97.0,95.0,36.11,98.0,75.33,62.0,19.0,24.0,...,36.11,0.0,98.0,98.0,98.0,0.000000,98.0,98.0,98.0,0.000000
2,2,p000001,89.0,99.0,36.11,122.0,86.00,62.0,22.0,24.0,...,36.11,0.0,106.0,98.0,122.0,13.856406,106.0,98.0,122.0,13.856406
3,3,p000001,90.0,95.0,36.11,122.0,86.00,62.0,30.0,24.0,...,36.11,0.0,114.0,98.0,122.0,13.856406,110.0,98.0,122.0,13.856406
4,4,p000001,103.0,88.5,36.11,122.0,91.33,62.0,24.5,24.0,...,36.11,0.0,122.0,122.0,122.0,0.000000,112.4,98.0,122.0,13.145341


In [4]:
import joblib
import os
import sys

# project_root=os.path.abspath(os.path.join(os.path.dirname(__file__),'..'))
# sys.path.append(project_root)
# save_dir = os.path.join(project_root, "saved_models")

df = pd.read_sql_query(
    "SELECT * FROM feature_table",
    conn
)
avg=df.mean(numeric_only=True)
joblib.dump(
    avg,
    os.path.join(r'C:\Users\KRISH JHA\OneDrive\Desktop\Sepsis\model\saved_models', "avg_columns.pkl")
    )
print(df.mean(numeric_only=True))

False                  776104.500000
HR                         84.369239
O2Sat                      97.171786
Temp                       36.857923
SBP                       123.722534
                           ...      
SBP_Rolling_Std_3h          7.748767
SBP_Rolling_Mean_6h       123.805125
SBP_Rolling_Min_6h        112.151204
SBP_Rolling_Max_6h        135.904120
SBP_Rolling_Std_6h          9.281613
Length: 109, dtype: float64


In [10]:
import os
import joblib

ids = pd.read_sql_query("""
SELECT DISTINCT Patient_id
FROM feature_table;
""", conn)
joblib.dump(ids, os.path.join(r'C:\Users\KRISH JHA\OneDrive\Desktop\Sepsis\model\saved_models','all_patient_ids.pkl'))

print(ids)

      Patient_id
0        p000001
1        p000002
2        p000003
3        p000004
4        p000005
...          ...
40331    p119996
40332    p119997
40333    p119998
40334    p119999
40335    p120000

[40336 rows x 1 columns]


In [18]:
import os
import joblib

data = pd.read_sql_query("""
SELECT * 
FROM feature_table
GROUP BY Patient_id;
""", conn)

joblib.dump(data, os.path.join(r'C:\Users\KRISH JHA\OneDrive\Desktop\Sepsis\model\saved_models','patient_data.pkl'))

data.head(10)

,False,Patient_id,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,BaseExcess,...,Temp_Rolling_Max_6h,Temp_Rolling_Std_6h,SBP_Rolling_Mean_3h,SBP_Rolling_Min_3h,SBP_Rolling_Max_3h,SBP_Rolling_Std_3h,SBP_Rolling_Mean_6h,SBP_Rolling_Min_6h,SBP_Rolling_Max_6h,SBP_Rolling_Std_6h
0,0,p000001,97.0,95.0,36.11,98.0,75.33,62.0,19.0,24.0,...,36.11,0.0,98.0,98.0,98.0,0.0,98.0,98.0,98.0,0.0
1,54,p000002,61.0,99.0,36.44,124.0,65.00,43.0,17.5,0.0,...,36.44,0.0,124.0,124.0,124.0,0.0,124.0,124.0,124.0,0.0
2,77,p000003,87.0,96.0,37.11,131.0,70.33,50.0,29.0,8.0,...,37.11,0.0,131.0,131.0,131.0,0.0,131.0,131.0,131.0,0.0
3,125,p000004,103.5,97.0,36.78,107.5,70.50,52.0,18.0,0.0,...,36.78,0.0,107.5,107.5,107.5,0.0,107.5,107.5,107.5,0.0
4,154,p000005,84.0,97.5,37.28,140.5,94.50,62.0,17.5,0.0,...,37.28,0.0,140.5,140.5,140.5,0.0,140.5,140.5,140.5,0.0
5,202,p000006,109.0,100.0,36.56,118.0,86.00,62.0,18.5,0.0,...,36.56,0.0,118.0,118.0,118.0,0.0,118.0,118.0,118.0,0.0
6,219,p000007,155.5,94.5,37.56,147.5,102.00,82.0,33.0,-12.0,...,37.56,0.0,147.5,147.5,147.5,0.0,147.5,147.5,147.5,0.0
7,264,p000008,73.0,99.0,36.06,100.0,67.00,49.5,16.5,-8.0,...,36.06,0.0,100.0,100.0,100.0,0.0,100.0,100.0,100.0,0.0
8,304,p000009,117.0,99.0,36.00,116.0,97.00,81.0,20.0,-7.0,...,36.00,0.0,116.0,116.0,116.0,0.0,116.0,116.0,116.0,0.0
9,562,p000010,82.0,100.0,35.50,112.0,79.50,63.0,14.0,0.0,...,35.50,0.0,112.0,112.0,112.0,0.0,112.0,112.0,112.0,0.0
